In [3]:
pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.9 MB 4.2 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.9 MB 4.2 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/9.9 MB 5.5 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 6.3 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.9 MB 6.7 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.9 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.9 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 6.0 MB/s  0:00:01

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   -----------------

In [4]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go

In [11]:
# ====== Load data from csv ======
df_raw = pd.read_excel("JunLiang CAP02_Segmented.xlsx", sheet_name="Gait marker", header=2, engine="openpyxl")

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [12]:
# ====== 2. Extract markers  ======
def extract_triplet(df, marker_token):
    idx = None
    for i, c in enumerate(df.columns):
        if isinstance(c, str) and marker_token.lower() in c.lower():
            idx = i
            break
    if idx is None:
        return None
    triplet_cols = df.columns[idx:idx+3]
    sub = df.loc[:, triplet_cols].copy()
    for c in triplet_cols:
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
    sub = sub.dropna(how="any")
    arr = sub.to_numpy(dtype=float)
    if arr.shape[1] != 3:
        return None
    return arr

markers_to_extract = {
    "Shin1": "KNEE MODEL:SHIN1",
    "Shin2": "KNEE MODEL:SHIN2",
    "Shin3": "KNEE MODEL:SHIN3",
    "Thigh1": "KNEE MODEL:THIGH1",
    "Thigh2": "KNEE MODEL:THIGH2",
    "Thigh3": "KNEE MODEL:THIGH3",
}

data = {}
for key, token in markers_to_extract.items():
    arr = extract_triplet(df_raw, token)
    if arr is None:
        print(f"❌ Could not parse triplet for {key} (token: {token})")
    else:
        data[key] = arr
        print(f"Parsed {key}: shape {arr.shape}")

N = min(arr.shape[0] for arr in data.values())
for k in data:
    data[k] = data[k][:N, :]


NameError: name 'df_raw' is not defined

In [13]:
# ====== 3. Compute frame and coordinates  ======
def normalize_rows(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n < 1e-12] = 1.0
    return v / n

def build_frame(p1, p2, p3):
    x = normalize_rows(p2 - p1)
    z = normalize_rows(np.cross(x, p3 - p1))
    y = normalize_rows(np.cross(z, x))
    R = np.stack([x, y, z], axis=2)
    O = p1
    return R, O

def sphere_fit(points):
    P = points
    b = np.sum(P * P, axis=1).reshape(-1, 1)
    A = np.hstack([2 * P, np.ones((P.shape[0], 1))])
    x, *_ = np.linalg.lstsq(A, b, rcond=None)
    c = x[:3, 0]
    r = float(np.sqrt(np.mean(np.sum((P - c) ** 2, axis=1))))
    return c, r

R_shin,  O_shin  = build_frame(data["Shin1"],  data["Shin2"],  data["Shin3"])
R_thigh, O_thigh = build_frame(data["Thigh1"], data["Thigh2"], data["Thigh3"])

KeyError: 'Shin1'

In [14]:
# ====== SPHERE FIT (COR) This one ======
thigh_cloud = np.vstack([data["Thigh1"], data["Thigh2"], data["Thigh3"]])
center, r = sphere_fit(thigh_cloud)
frame = N // 2
shinOrigin  = O_shin[frame]
thighOrigin = O_thigh[frame]
Rsh = R_shin[frame]
Rth = R_thigh[frame]

deltaShin  = center - shinOrigin
deltaThigh = center - thighOrigin
distanceShinCOR  = float(np.linalg.norm(deltaShin))
distanceThighCOR = float(np.linalg.norm(deltaThigh))
center_in_shin  = Rsh.T @ (center - shinOrigin)
center_in_thigh = Rth.T @ (center - thighOrigin)

KeyError: 'Thigh1'

In [15]:
# ====== 5. PRINT SUMMARY ======
print("=== Numeric Summary (parsed triplets) ===")
print(f"Frames used: {N}")
print(f"Sphere-fit COR center (lab): {center}")
print(f"Sphere-fit radius (mm): {r:.3f}")
print(f"Distance (Thigh origin → COR): {distanceThighCOR:.3f} mm")
print(f"Distance (Shin origin  → COR): {distanceShinCOR:.3f} mm")
print(f"Δ (Thigh → COR) [X,Y,Z] (mm): {deltaThigh}")
print(f"Δ (Shin  → COR) [X,Y,Z] (mm): {deltaShin}")
print(f"COR in Thigh local frame (mm): {center_in_thigh}")
print(f"COR in Shin  local frame (mm): {center_in_shin}")

=== Numeric Summary (parsed triplets) ===


NameError: name 'N' is not defined

In [16]:
# ====== 6. 3D VISUALIZATION ======
def trace_points(name, pts, color, size=3):
    return go.Scatter3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2],
        mode='markers', marker=dict(size=size, color=color),
        name=name, opacity=0.8
    )

def trace_arrow(name, origin, vec, color, width=6):
    p2 = origin + vec
    return go.Scatter3d(
        x=[origin[0], p2[0]],
        y=[origin[1], p2[1]],
        z=[origin[2], p2[2]],
        mode='lines', line=dict(width=width, color=color),
        name=name
    )

def sphere_surface(center, radius, res=24):
    u = np.linspace(0, 2*np.pi, res)
    v = np.linspace(0, np.pi, res)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones_like(u), np.cos(v))
    return go.Surface(x=x, y=y, z=z, opacity=0.25, showscale=False, name='COR sphere')

traces = []
# Markers
traces += [trace_points("Thigh1", data["Thigh1"], "red", size=2)]
traces += [trace_points("Thigh2", data["Thigh2"], "red", size=2)]
traces += [trace_points("Thigh3", data["Thigh3"], "red", size=2)]
traces += [trace_points("Shin1",  data["Shin1"],  "blue", size=2)]
traces += [trace_points("Shin2",  data["Shin2"],  "blue", size=2)]
traces += [trace_points("Shin3",  data["Shin3"],  "blue", size=2)]

scale = 40.0
# Shin and Thigh triads
traces += [
    trace_arrow("Shin X",  shinOrigin,  scale * Rsh[:,0], "red"),
    trace_arrow("Shin Y",  shinOrigin,  scale * Rsh[:,1], "green"),
    trace_arrow("Shin Z",  shinOrigin,  scale * Rsh[:,2], "blue"),
    trace_arrow("Thigh X", thighOrigin, scale * Rth[:,0], "orange"),
    trace_arrow("Thigh Y", thighOrigin, scale * Rth[:,1], "brown"),
    trace_arrow("Thigh Z", thighOrigin, scale * Rth[:,2], "purple"),
]

# COR and sphere
traces += [go.Scatter3d(x=[center[0]], y=[center[1]], z=[center[2]],
                        mode='markers+text',
                        marker=dict(size=5, color='black'),
                        text=["COR"],
                        textposition="top center",
                        name="COR")]
traces += [sphere_surface(center, r)]

# Connecting lines and per-axis components (Shin)
traces += [
    trace_arrow("Shin→COR",  shinOrigin,  center - shinOrigin,  "black", width=3),
    trace_arrow("Thigh→COR", thighOrigin, center - thighOrigin, "brown", width=3),
    trace_arrow("ΔX shin", shinOrigin, np.array([deltaShin[0], 0, 0]), "red"),
    trace_arrow("ΔY shin", shinOrigin, np.array([0, deltaShin[1], 0]), "green"),
    trace_arrow("ΔZ shin", shinOrigin, np.array([0, 0, deltaShin[2]]), "blue"),
]

layout = go.Layout(
    scene=dict(
        xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
        aspectmode='data'
    ),
    title="COR fit, Shin/Thigh frames, and distances",
    showlegend=True, height=800
)

fig = go.Figure(data=traces, layout=layout)
fig.show()


KeyError: 'Thigh1'

### Visualise CORFemur markers

This section loads the dynamic trial CSV (`data/CORDynamic2.csv`) and plots the trajectories of the following markers in 3D:

- CORFemur:CentrePlateMarker
- CORFemur:F1
- CORFemur:F2
- CORFemur:F3
- CORFemur:F4
- CORFemur:B1
- CORFemur:B2
- CORFemur:B3

Run the next cell to generate the 3D plot of the marker trajectories.

In [17]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [20]:
# Visualize CORFemur marker trajectories (3D) — robust header parsing
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

markers = [
    'CORFemur:CentrePlateMarker',
    'CORFemur:F1','CORFemur:F2','CORFemur:F3','CORFemur:F4',
    'CORFemur:B1','CORFemur:B2','CORFemur:B3'
]

csv_path = Path('data/CORDynamic2.csv')
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path.resolve()}")

text = csv_path.read_text(encoding='utf-8')
lines = text.splitlines()

In [21]:
# find the header row that defines column tokens (looks for 'Frame,' at start)
header_idx = None
for i, ln in enumerate(lines):
    if ln.startswith('Frame,') or ln.startswith('Frame,Sub Frame') or ln.startswith('Frame,Sub'):
        header_idx = i
        break
if header_idx is None:
    raise ValueError("Could not find header row starting with 'Frame,' in the CSV")

In [24]:
# Assume marker row is immediately before the 'Frame' header
marker_row_idx = header_idx - 1
# Optionally there's a units row immediately after header (e.g. ',,mm,mm,...')
units_row_idx = header_idx + 1 if header_idx + 1 < len(lines) else None

def split_and_pad(ln, length):
    parts = ln.split(',')
    if len(parts) < length:
        parts += [''] * (length - len(parts))
    return parts

# Read the three candidate header lines
marker_parts = lines[marker_row_idx].split(',')
colname_parts = lines[header_idx].split(',')
units_parts = []
if units_row_idx is not None:
    units_parts = lines[units_row_idx].split(',')

maxlen = max(len(marker_parts), len(colname_parts), len(units_parts) if units_parts else 0)
if maxlen == 0:
    raise RuntimeError('Empty header rows encountered')

marker_parts = split_and_pad(lines[marker_row_idx], maxlen)
colname_parts = split_and_pad(lines[header_idx], maxlen)
units_parts = split_and_pad(lines[units_row_idx], maxlen) if units_parts else [''] * maxlen

# Build 2-level tuples (marker, colname). Trim whitespace.
col_tuples = [(marker_parts[i].strip(), colname_parts[i].strip()) for i in range(maxlen)]

# Determine where the numeric data starts: if a units row exists and contains 'mm' or similar, skip that too
data_start_row = header_idx + 1
if any(u.strip().lower() in ('mm', 'm', 'cm') for u in units_parts):
    data_start_row = units_row_idx + 1

# Read the data as raw (no header), then assign columns
# Use engine='python' and read as strings to avoid mixed-type inference problems; set low_memory=False for consistency
try:
    df_data = pd.read_csv(csv_path, header=None, skiprows=data_start_row, encoding='utf-8', sep=',', engine='python', dtype=str, na_values=['', 'NA'], keep_default_na=False, low_memory=False)
except Exception as e:
    raise RuntimeError(f"Failed to read CSV body: {e}")

# Trim or extend columns to match col_tuples length
if df_data.shape[1] > len(col_tuples):
    df_data = df_data.iloc[:, :len(col_tuples)]
elif df_data.shape[1] < len(col_tuples):
    # pad missing columns with NaN
    for j in range(df_data.shape[1], len(col_tuples)):
        df_data[j] = pd.NA

# Assign MultiIndex columns
try:
    df_data.columns = pd.MultiIndex.from_tuples(col_tuples[: df_data.shape[1]])
except Exception as e:
    raise RuntimeError(f"Failed to construct MultiIndex columns: {e}")

# Convert numeric-looking columns to numeric now where needed (we'll convert per marker when plotting)

# Replace empty-string entries with NaN for numeric conversions later
# (keep original values as strings for debugging if needed)

df = df_data.copy()
# Find the frame column (second-level name 'Frame')
frame_cols = [col for col in df.columns if str(col[1]).strip().lower() == 'frame']
if not frame_cols:
    # Fallback: look for top-level empty and second-level 'Frame' or first column
    possible = [col for col in df.columns if str(col[0]).strip() == '' and str(col[1]).strip().lower() == 'frame']
    if possible:
        frame_col = possible[0]
    else:
        # as last resort assume first column
        frame_col = df.columns[0]
else:
    frame_col = frame_cols[0]

ParserError: Error tokenizing data. C error: Expected 20 fields in line 8659, saw 27


In [ ]:
# Convert frame to numeric and filter non-numeric rows
df_frame = pd.to_numeric(df[frame_col], errors='coerce')
if df_frame.isna().all():
    # If frame column parsed as NaN (maybe a header line remained), try to coerce first column as integer index
    try:
        df.index = range(len(df))
        df_frame = pd.Series(range(len(df)))
    except Exception:
        raise RuntimeError('Could not determine numeric frame values from file')

# Keep only rows with numeric frame (if any)
numeric_mask = df_frame.notna()
if numeric_mask.any():
    df = df.loc[numeric_mask, :].copy()
    df[frame_col] = df_frame.loc[numeric_mask].astype(int)

# Plot trajectories for each marker
fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
found_any = False
for m in markers:
    # find X/Y/Z columns for marker m (match top-level header to marker name)
    cols = {axis: col for col in df.columns for axis in ['X', 'Y', 'Z']
            if str(col[0]).strip() == m and str(col[1]).strip() == axis}
    if len(cols) != 3:
        print(f"Warning: incomplete XYZ for marker '{m}' (found: {list(cols.keys())}). Skipping.")
        continue
    x = pd.to_numeric(df[cols['X']], errors='coerce')
    y = pd.to_numeric(df[cols['Y']], errors='coerce')
    z = pd.to_numeric(df[cols['Z']], errors='coerce')
    ax.plot(x, y, z, label=m)
    # mark start and end
    if len(x):
        ax.scatter(x.iloc[0], y.iloc[0], z.iloc[0], marker='o', s=30)
        ax.scatter(x.iloc[-1], y.iloc[-1], z.iloc[-1], marker='x', s=40)
    found_any = True

if not found_any:
    print('No marker trajectories plotted. Check marker names and CSV header format.')

ax.set_xlabel('X (mm)')
ax.set_ylabel('Y (mm)')
ax.set_zlabel('Z (mm)')
ax.set_title('CORFemur marker trajectories')
ax.legend()
plt.tight_layout()
plt.show()